In [7]:
import json
from datetime import datetime, timedelta 


def session_abr(session_name):
    abr_dict = {
        "Practice 1":"FP1",
        "Practice 2":"FP2",
        "Practice 3":"FP3",
        "Qualifying":"Q",
        "Sprint Qualifying":"SQ",
        "Sprint":"Sprint",
        "Race":"Grand Prix",
        "Chequered Flag":"Flag"
    }
    abr = abr_dict[session_name]
    if abr:
        return abr
    return session_name

    
def event_name(event):
    value = event.replace(" Grand Prix", "")
    match value:
        case "Australia":
            result = "Melbourne"
        case "China":
            result = "Shanghai"
        case "Japan":
            result = "Suzuka"
        case "Bahrain":
            result = "Bahrain"
        case "Saudi Arabia":
            result = "Jeddah"
        case "Emilia Romagna":
            result = "Imola"
        case "Spain":
            result = "Barcelona"
        case "Canada":
            result = "Montreal"
        case "Austrian":
            result = "Red Bull"
        case "British":
            result = "Silverstone"
        case "Belgian":
            result = "Spa"
        case "Hungarian":
            result = "Hungary"
        case "Italy":
            result = "Monza"
        case "United States":
            result = "Austin"
        case "S\u00e3o Paulo":
            result = "Brazil"
        case "United Arab Emirates":
            result = "Abu Dhabi"
        case "Barcelona Catalunya":
            result = "Barcelona Catalunya"
        case _:
            result = value
    return result

def month_as_num(month_name):
    months = {
        "Jan": '01',
        "Feb": '02',
        "Mar": '03',
        "Apr": '04',
        "May": '05',
        "Jun": '06',
        "Jul": '07',
        "Aug": '08',
        "Sep": '09',
        "Oct": '10',
        "Nov": '11',
        "Dec": '12'
    }
    return months.get(month_name, 0)

def practice_events():
    events = [
        {
            "event": "Sakhir",
            "session": "Winter",
            "start": "2026-02-11T09:30:00-00:00"
        },
        {
            "event": "Sakhir",
            "session": "Winter",
            "start": "2026-02-11T13:00:00-00:00"
        },
        {
            "event": "Sakhir",
            "session": "Winter",
            "start": "2026-02-12T09:30:00-00:00"
        },
        {
            "event": "Sakhir",
            "session": "Winter",
            "start": "2026-02-12T13:00:00-00:00"
        },
        {
            "event": "Sakhir",
            "session": "Winter",
            "start": "2026-02-13T09:30:00-00:00"
        },
        {
            "event": "Sakhir",
            "session": "Winter",
            "start": "2026-02-13T13:00:00-00:00"
        },
        {
            "event": "Sakhir",
            "session": "Winter",
            "start": "2026-02-18T09:30:00-00:00"
        },
        {
            "event": "Sakhir",
            "session": "Winter",
            "start": "2026-02-18T13:00:00-00:00"
        },
        {
            "event": "Sakhir",
            "session": "Winter",
            "start": "2026-02-19T09:30:00-00:00"
        },
        {
            "event": "Sakhir",
            "session": "Winter",
            "start": "2026-02-19T13:00:00-00:00"
        },
        {
            "event": "Sakhir",
            "session": "Winter",
            "start": "2026-02-20T09:30:00-00:00"
        },
        {
            "event": "Sakhir",
            "session": "Winter",
            "start": "2026-02-20T13:00:00-00:00"
        },
        {
            "event": "Melbourne",
            "session": "FP1",
            "start": "2026-03-05T09:30:00-00:00"
        },
        {
            "event": "Melbourne",
            "session": "FP2",
            "start": "2026-03-05T13:00:00-00:00"
        },
        {
            "event": "Melbourne",
            "session": "FP3",
            "start": "2026-03-06T10:30:00-00:00"
        },
        {
            "event": "Melbourne",
            "session": "Q",
            "start": "2026-03-06T14:00:00-00:00"
        },
        {
            "event": "Melbourne",
            "session": "Grand Prix",
            "start": "2026-03-07T13:00:00-00:00"
        }
    ]
    return events

# print(practice_events())

In [ ]:
#https://beautiful-soup-4.readthedocs.io/en/latest/#kinds-of-objects

import requests
from bs4 import BeautifulSoup

# year = 2026

# Send an HTTP request to the website
def get_f1_schedule(year):
    events = []
    print("Getting details for winter practice")
    events.append(practice_events())
    base_url = "https://www.formula1.com"
    url = "{}/en/racing/{}".format(base_url, year)
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    title = soup.title.string
    print(title)
    rows = soup.find_all('a', class_='group')
    print(f"Found {len(rows)} events")
    for row in rows:
        if row['href'] == "/en/racing/2026/pre-season-testing-2" or row['href'] == "/en/racing/2026/pre-season-testing-1":
            print("Found pre season testing, skipping")
            continue
        details = get_f1_event_details(year, row['href'])
        events.append(details)

    flat = [item for sublist in events for item in sublist]
    return flat


def get_f1_event_details(year, url):
    print("Getting details for {}".format(url))
    event = url.split("/")[-1].replace("-", " ").title()
    event = event_name(event)
    base_url = "https://www.formula1.com"
    response = requests.get("{}{}".format(base_url, url))
    soup = BeautifulSoup(response.content, 'html.parser')
    rows = soup.find('ul').find_all('li')
    details = []
    for row in rows:
        spans = row.find_all('span')
        # print("<-------{}-------->".format(len(spans)))
        # print(row)
        day = spans[1].text
        month = month_as_num(spans[2].text)
        session = session_abr(spans[4].text)
        time = spans[7].text
        start = time.split(" - ")[0]
        session_details = {
            "event": event,
            "session": session,
            "start": "{}-{}-{}T{}:00-00:00".format(year, month, day, start)
        }
        details.append(session_details)
    return details


# details = get_f1_schedule(2026)
# print(details)
# details = get_f1_event_details("/en/racing/2026/australia")
# print(details)


Getting details for winter practice
F1 Schedule 2026 - Official Calendar of Grand Prix Races
Found 26 events
Found pre season testing, skipping
Found pre season testing, skipping
Getting details for /en/racing/2026/australia
Getting details for /en/racing/2026/china
Getting details for /en/racing/2026/japan
Getting details for /en/racing/2026/bahrain
Getting details for /en/racing/2026/saudi-arabia
Getting details for /en/racing/2026/miami
Getting details for /en/racing/2026/canada
Getting details for /en/racing/2026/monaco
Getting details for /en/racing/2026/barcelona-catalunya
Getting details for /en/racing/2026/austria
Getting details for /en/racing/2026/great-britain
Getting details for /en/racing/2026/belgium
Getting details for /en/racing/2026/hungary
Getting details for /en/racing/2026/netherlands
Getting details for /en/racing/2026/italy
Getting details for /en/racing/2026/spain
Getting details for /en/racing/2026/azerbaijan
Getting details for /en/racing/2026/singapore
Getting

In [3]:
import json
year = 2026

sessions = get_f1_schedule(year)
# print(sessions)
# sessions = practice_events()

# Serialize the data to a JSON formatted string
json_string = json.dumps(sessions, indent=4)

# Write the JSON string to a file
with open(f"{year}_schedule.json", "w") as outfile:
    outfile.write(json_string)



Getting details for winter practice
F1 Schedule 2026 - Official Calendar of Grand Prix Races
Found 26 events
Getting details for /en/racing/2026/pre-season-testing-1
Getting details for /en/racing/2026/pre-season-testing-2


KeyError: 'Day 1'